# Iterative Tweet Generation Workflow with LangGraph

This notebook implements an iterative generator-critic workflow in LangGraph. 
- The **Generator** (`generate_tweet`) drafts and refines a tweet.
- The **Critic** (`critique_tweet`) reviews the draft and provides feedback.
- A **Router** (`should_continue`) loops back to the generator if the critic is not satisfied (up to a maximum number of iterations).

In [ ]:
import os
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# Ensure your OpenAI API key is set
# os.environ["OPENAI_API_KEY"] = "your-api-key-here"

In [9]:
# 1. Define the State schema
class TweetState(TypedDict):
    topic: str
    tweet: str
    critique: str
    iteration: int

In [10]:
# 2. Define the Generator Node
def generate_tweet(state: TweetState):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    topic = state["topic"]
    tweet = state.get("tweet", "")
    critique = state.get("critique", "")
    iteration = state.get("iteration", 0)
    
    if iteration == 0:
        # First draft
        messages = [
            SystemMessage(content="You are a professional social media manager. Write an engaging tweet about the given topic. Keep it strictly under 280 characters, include a few relevant hashtags, and capture attention."),
            HumanMessage(content=f"Topic: {topic}")
        ]
    else:
        # Refined draft
        messages = [
            SystemMessage(content="You are a professional social media manager. Refine the previous tweet draft based on the critique feedback. Make sure it stays under 280 characters."),
            HumanMessage(content=f"Topic: {topic}\nPrevious Tweet: {tweet}\nFeedback/Critique: {critique}\n\nCreate an improved version addressing all feedback.")
        ]
        
    response = llm.invoke(messages)
    return {
        "tweet": response.content.strip(),
        "iteration": iteration + 1
    }

# 3. Define the Critic Node
def critique_tweet(state: TweetState):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)
    topic = state["topic"]
    tweet = state["tweet"]
    
    messages = [
        SystemMessage(content="""You are a tweet quality reviewer. 
Review the given tweet. It must be relevant to the topic, highly engaging, and strictly under 280 characters.
If the tweet is excellent and meets all requirements, reply with exactly 'APPROVED'.
Otherwise, provide clear, concise feedback on what needs to be improved."""),
        HumanMessage(content=f"Topic: {topic}\nTweet: {tweet}")
    ]
    
    response = llm.invoke(messages)
    return {"critique": response.content.strip()}

# 4. Define the Router Function
def should_continue(state: TweetState) -> Literal["generate", END]:
    # Stop if approved OR if we have completed 3 iterations (to prevent infinite loops)
    if state.get("critique") == "APPROVED" or state.get("iteration", 0) >= 3:
        return END
    return "generate"

In [11]:
# 5. Build and Compile the Graph
builder = StateGraph(TweetState)

# Add nodes
builder.add_node("generate", generate_tweet)
builder.add_node("critique", critique_tweet)

# Add edges
builder.add_edge(START, "generate")
builder.add_edge("generate", "critique")

# Add conditional routing edge from critique
builder.add_conditional_edges(
    "critique",
    should_continue,
   
)

workflow = builder.compile()

In [12]:
# 6. Test the workflow
# Note: Ensure you have loaded your OPENAI_API_KEY in your environment before running this cell
try:
    initial_state = {
        "topic": "The future of AI agentic coding assistants in software development",
        "iteration": 0
    }
    
    # Run the compiled workflow
    final_state = workflow.invoke(initial_state)
    
    print("--- Final Results ---")
    print(f"Total Iterations: {final_state['iteration']}")
    print(f"Critic's Final Verdict: {final_state['critique']}")
    print(f"Final Tweet:\n{final_state['tweet']}")
except Exception as e:
    print("Error running workflow:", e)
    print("Make sure you have set the OPENAI_API_KEY environment variable.")

Error running workflow: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.
Make sure you have set the OPENAI_API_KEY environment variable.
